# Farmland Classification 

We used Open Street Map (OSM) for downloading farmland data. This shapefile is used in the BaseMap Jupyter Notebook to classify commercial-scale farming. You can read more about OSM farmaland data on https://wiki.openstreetmap.org/wiki/Tag:landuse=farmland. 

The original OSM shapefile specifies the crop and the system (irrigated vs rainfed) for each polygon. This Jupyter Notebook serves to create an additional column that creates crop classes that creates consistency with the IFPRI dataset. 

We used Zambia as an example. For instance, if a polygon had sugarcane and irrigation, than the class becomes SUGC_HI. Since the classification can vary greately depending on the country and engagement with local experts, this part needs to be manually edited by the user to match the context-specific needs. 

Here you will find the step-by-step guide on how to download farmland data from OSM on QGIS, and the classification code (to be revised depending on the classification assumptions).

The final output is a shapefile named farmlands_reclassified in the 00.InputData folder.

### Download farmland data from OSM

#### Step 1: Install the QuickOSM Plugin
- Open QGIS.
- Select Plugins > Manage and Install Plugins
- Search for QuickOSM in the search bar.
- Click Install Plugin, then close the window.

Look for a small green magnifying glass icon  on the toolbar, or access it via Vector > QuickOSM > QuickOSM.

#### Step 2: Set Up the Query
- Open QuickOSM 
- Go to Quick query tab
- Set the parameters:
  Key: landuse
  Value: farmland
  
  Location Filter: 
  - Option 1 - In location: Type a town, region, or country name (e.g., Zambia).
  - Option 2 - Canvas Extent: Uses your current map view area in QGIS.

#### Output

The plug in returns a polygon, a vector, and a point layer.

**Export only the polygon** and rename it as *OSM_farmland*. **Make sure to choose the same projection system as your user settings in BaseMaps**

In [1]:
import os
import re
import warnings
from typing import Dict, Tuple, Optional, List

import geopandas as gpd
import pandas as pd


from pathlib import Path
from Map.reclassify_farmlands import list_unique_crop, reclassify_farmlands, DEFAULT_MAPPING




In [2]:
ROOT_DIR = os.path.abspath(os.curdir)
OUT_NODATA = 255
Data_dir = os.path.join(ROOT_DIR, 'Data')
Input_data = os.path.join(Data_dir, '00.InputData')



### 1) List unique crops found in the shapefile

OSM farmland attribute tables have two key columns: one named crop and one irrigation. This function lists all the crops in the uploaded shapefile. Since each shapefile varies depending on the country, it is important to update the classification rule based on the crop list. This can be done by editing the user mapping dictionary in step 2. 

In [3]:
# Print crop list
farmland_crop = list_unique_crop(str(Input_data))
print("Found the following unique crop strings in the `crop` column:")
for i, c in enumerate(farmland_crop, 1):
    print(f"{i:2d}. {c}")

Found the following unique crop strings in the `crop` column:
 1. Banana
 2. cassava
 3. cereal
 4. coffee
 5. corn
 6. grass
 7. maize
 8. native_pasture
 9. palm oil
10. soy
11. sugarcane
12. vegetable;rape;vegetables;Papaya;mango;Guave
13. wheat
14. wheat, soya
15. wheat, soybean
16. wheat,soyabean
17. wheat; soybean
18. wheat;soy
19. wheat;soy bean


## 2) Edit the classification rule in the USER_MAPPING. 
The user mapping is a dictionary structured as "crop name in the OSM list": ("class name if irrigated", "class name if rainfed"). We distinguish among irrigated and rainfed based on the second column in the shapefile called irrigation. This column can have a string called "pivot" or nothing. If there is written "pivot", it means that the farmland is irrigated. For instance, if in the dictionary there is written:
"sugarcane": ("SUGC_HI", "SUGC_HR"), this means that if the crop is sugarcane and "pivot" under the irrigation column, then it becomes SUGC_HI; if there is no pivot, then the class is SUCG_HR. If the cell has no crops, then it falls under the class Other.

Note that it is important to adopt a **consistent naming convention** with the one used by the **IFPRI** insitute (four upper case letter identifiers). You can check it in the 00.InputData/spam2020V2 folder. Then always keep HI and HR for irrigated and rainfed systems.

In [4]:
# 2) Prepare a user-editable mapping. 

USER_MAPPING = dict(DEFAULT_MAPPING)  # copy defaults; edit this cell to customize

# Example: to override sugarcane, set:
# USER_MAPPING['sugarcane'] = ('SUGC_HI', 'SUGC_HR')

print('\nCurrent USER_MAPPING keys (edit this cell to change):')
for k in sorted(USER_MAPPING.keys(), key=lambda x: ('' if x is None else str(x).lower())):
    print('-', k, '->', USER_MAPPING[k])



Current USER_MAPPING keys (edit this cell to change):
- None -> ('Other_HI', 'Other_HR')
- banana -> ('BANA_HI', 'BANA_HR')
- cassava -> ('CASS_HI', 'CASS_HR')
- cereal -> ('MILL_HI', 'MILL_HR')
- coffee -> ('COFF_HI', 'COFF_HR')
- corn -> ('MAIZ_HI', 'MAIZ_HR')
- maize -> ('MAIZ_HI', 'MAIZ_HR')
- palm -> ('SUNF_HR', 'SUNF_HR')
- palm oil -> ('SUNF_HR', 'SUNF_HR')
- soy -> ('SOYB_HI', 'SOYB_HR')
- soybean -> ('SOYB_HI', 'SOYB_HR')
- sugarcane -> ('SUGC_HI', 'SUGC_HR')
- vegetable -> ('VEGE_HI', 'VEGE_HR')
- vegetable;rape;vegetables;Papaya;mango;Guave -> ('VEGE_HI', 'VEGE_HR')
- vegetables -> ('VEGE_HI', 'VEGE_HR')
- wheat -> ('WHEA_HI', 'WHEA_HI')
- wheat, soya -> ('WHEA_HI', 'SOYB_HR')
- wheat, soyabean -> ('WHEA_HI', 'SOYB_HR')
- wheat, soybean -> ('WHEA_HI', 'SOYB_HR')
- wheat,soyabean -> ('WHEA_HI', 'SOYB_HR')
- wheat; soybean -> ('WHEA_HI', 'SOYB_HR')
- wheat;soy -> ('WHEA_HI', 'SOYB_HR')
- wheat;soy bean -> ('WHEA_HI', 'SOYB_HR')
- wheat;soybean -> ('WHEA_HI', 'SOYB_HR')


Fill or edit the `USER_MAPPING` dictionary in the previous cell so the values
match your desired crop class names. Keys should match crop names
as they appear above.

When ready, run the next cell to create `farmlands_reclassified.shp` in the
`Data/00.InputData` folder. This is the file that will be then used in the BaseMap workflow. 


In [ ]:
# 3) Run the reclassification and write output shapefile
gdf = reclassify_farmlands(str(Input_data), mapping=USER_MAPPING, output_shp='farmlands_reclassified.shp')
print("Wrote reclassified shapefile:", os.path.join(Input_data, 'farmlands_reclassified.shp'))
print("Rows:", len(gdf))

# show a sample of the new column
if 'crop_class' in gdf.columns:
    print("Sample of the new `crop_class` column:")
    display(gdf[['crop', 'irrigation', 'crop_class']].head(10))
else:
    print('Output column `crop_class` not found in result.')


Wrote reclassified shapefile: g:\My Drive\01.KTH-PhD\0.Papers\02.Paper\AgrMap-fix-bug\AgrMap-fix-bug\Data\00.InputData\farmlands_reclassified.shp
Rows: 8304


,crop,irrigation,crop_class
0,None,pivot,Other_HI
1,None,pivot,Other_HI
2,None,None,Other_HR
3,None,None,Other_HR
4,None,None,Other_HR
5,None,None,Other_HR
6,sugarcane,None,SUGC_HR
7,"wheat, soybean",pivot,WHEA_HI
8,None,None,Other_HR
9,None,None,Other_HR
